In [0]:
data = [
    (1001, "Laptop", "North", 2, 800),
    (1002, "Mobile", "South", 3, 400),
    (1003, "Keyboard", "East", 5, 50),
    (1004, "Monitor", "West", 2, 300),
    (1005, "Tablet", None, 1, 300),
    (1002, "Mobile", "South", 3, 400)
]

columns = [
    "order_id",
    "product",
    "region",
    "quantity",    
    "unit_price"
]


orders_df=spark.createDataFrame(data,columns)


In [0]:
selected_orders = orders_df.select(
    "order_id",
    "product",
    "quantity",
    "unit_price"
)

display(selected_orders)

In [0]:
unique_products=orders_df.select(
"product"
).distinct()
display(unique_products)

In [0]:
from pyspark.sql import functions as F
orders_with_revenue = orders_df.withColumn(
    "revenue",
    F.col("quantity") * F.col("unit_price")
)

display(orders_with_revenue)

In [0]:
regional_revenue=orders_with_revenue.groupBy("region"
                                             ).agg(F.sum("revenue").alias("total_revenue"))

display(regional_revenue)                               

In [0]:
regional_revenue.orderBy(
    F.col("total_revenue").desc()
).show()

In [0]:
total_revenue = orders_with_revenue.agg(
    F.sum("revenue").alias("grand_total")
)

display(total_revenue)

In [0]:
regional_contribution=regional_revenue.withColumn(
    "revenue_percentage",
  F.round((F.col("total_revenue")/5150)*100,2)
)
display(regional_contribution)


In [0]:
grand_total=orders_with_revenue.agg(
F.sum("revenue")).collect()[0][0]
print(grand_total)

In [0]:
regional_contribution = regional_revenue.withColumn(
    "revenue_percentage",
    F.round(
        (F.col("total_revenue") / grand_total) * 100,
        2
    )
)

display(regional_contribution)

In [0]:

display(regional_contribution.orderBy(
    F.col("revenue_percentage").desc()
))

### Handle missing regions


In [0]:
orders_cleaned=orders_with_revenue.fillna({
    "region":"Unknown"
})
display(orders_cleaned)

In [0]:
duplicate_records=orders_cleaned.groupBy(
orders_cleaned.columns
).count().filter(F.col("count")>1)

display(duplicate_records)

In [0]:

orders_deduplicated=orders_cleaned.dropDuplicates()
display(orders_deduplicated)

In [0]:
print("Records before:", orders_cleaned.count())
print("Records after:", orders_deduplicated.count())